# 02. Camada Gold e Auditoria de Não-Vazamento Temporal (Zero Data Leakage)
**Artefato Analítico-Preditivo para Planejamento de Demanda e Mitigação de Desperdício**  
*Mestrado Profissional em Educação Profissional e Tecnológica (ProfEPT) / Gestão Pública*  
**Pesquisador:** Michel Silva de Souza  

---

## Storytelling: A Prova Matemática do Isolamento Temporal
Em projetos de aprendizado de máquina aplicados a séries temporais, o erro mais comum e perigoso na literatura é o **Vazamento de Dados (Data Leakage)**: quando o modelo, sem que o cientista perceba, utiliza direta ou indiretamente informações do futuro durante a fase de treino ou validação.

### Por que a Validação Cruzada Aleatória (K-Fold) Falha?
Se embaralharmos dias letivos aleatoriamente em folds, uma terça-feira cairá no treino e a segunda-feira anterior cairá no teste. O algoritmo aprende "olhando para trás", mas na vida real, na sexta-feira anterior, a nutricionista não conhece a segunda-feira!

### O Protocolo Operacional Estrito do Nosso Artefato:
1. **Marco de Corte (`origem`):** Toda previsão é calculada como se estivéssemos fisicamente na **sexta-feira à tarde**, usando estritamente o histórico de refeições até aquela data (`data < origem`).
2. **Dois Horizontes Reais:**
   - **$T+1$ (Preparo - Semana Seguinte):** 3 a 7 dias de antecedência. Permite usar o cardápio publicado e a estimativa humana da nutricionista.
   - **$T+2$ (Compras Quinzenais):** 10 a 14 dias de antecedência. Modelo **100% autônomo**, proibido de utilizar estimativas humanas de curto prazo.
3. **Auditoria Adversarial:** Embaralhamos os consumos futuros e provamos matematicamente que nenhum preditor do passado sofre alteração.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.core.config import GOLD, RESULTADOS
from src.core.audit_leakage import executar_auditoria_completa
from src.core.terminology import NOMES_HORIZONTES

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120


### 1. Carregamento dos Datasets Gold Auditados ($T+1$ e $T+2$)
Cada linha representa uma previsão para um dia letivo futuro, calculada a partir de sua respectiva sexta-feira de corte.

In [ ]:
t1 = pd.read_parquet(GOLD / "gold_t1.parquet")
t2 = pd.read_parquet(GOLD / "gold_t2.parquet")

print(f"Horizonte T+1 (Preparo): {t1.shape[0]} linhas x {t1.shape[1]} colunas | {t1['origem'].min().strftime('%d/%m/%Y')} a {t1['origem'].max().strftime('%d/%m/%Y')}")
print(f"Horizonte T+2 (Compras): {t2.shape[0]} linhas x {t2.shape[1]} colunas | {t2['origem'].min().strftime('%d/%m/%Y')} a {t2['origem'].max().strftime('%d/%m/%Y')}")


### 2. Dicionário de Atributos da Camada Gold
Relação auditável de grupos de variáveis disponíveis em cada horizonte operacional.

In [ ]:
dicionario = pd.read_csv(GOLD / "gold_dicionario_atributos.csv", sep=";")
dicionario.groupby("grupo")[["coluna"]].count().rename(columns={"coluna": "Quantidade de Atributos"}).join(
    dicionario.groupby("grupo")["disponivel_em"].first()
)


### 3. Execução da Auditoria de Não-Vazamento Temporal
Executamos o teste adversarial que:
1. Valida se todas as origens são sextas-feiras e as antecedências respeitam as restrições operacionais.
2. Confirma que dados humanos não vazaram para o horizonte de compras ($T+2$).
3. Embaralha consumos futuros com ruído aleatório e verifica a igualdade de frames dos preditores.

In [ ]:
relatorio_auditoria = executar_auditoria_completa()

for teste in relatorio_auditoria["testes"]:
    status = "APROVADO [OK]" if teste["sucesso"] else "REPROVADO [FALHA]"
    print(f"{status}: {teste['teste']}")
    if "detalhe" in teste:
        print(f"   -> {teste['detalhe']}")
    if "detalhes" in teste:
        for d in teste["detalhes"]:
            print(f"   * {d}")


### 4. Visualização do Isolamento Temporal
Este gráfico ilustra a relação entre o momento em que a decisão é tomada (**Sexta-feira de Emissão**) e a data em que a refeição é servida (**Dia Previsto**), comprovando a antecedência estrita.

In [ ]:
amostra_t1 = t1.tail(20)

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(amostra_t1["origem"], amostra_t1["antecedencia_dias"], color="#2980b9", s=60, label="Previsões T+1 (Preparo: 3 a 7 dias)")
ax.set_title("Antecedência Operacional Estrita das Emissões (Sextas-feiras)", fontsize=13, fontweight="bold")
ax.set_xlabel("Sexta-feira de Corte (Origem)")
ax.set_ylabel("Dias de Antecedência até o Almoço")
ax.set_yticks(range(3, 8))
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
